# Build a Voice Live speech agent and run a recorded audio prompt through STT, agent, and TTS

Your customer-support team wants to test a voice agent before greenlighting a phone-based pilot. The founder is worried Voice Live is going to be flaky because the docs explicitly say preview. You have one session to wire a Voice Live agent in Foundry, run a 12-second recorded audio prompt through STT, agent, and TTS, and prove the loop produces a response audio file the team can listen to. The deliverable is a WAV file plus a transcript of the original question; the founder can decide on the phone pilot after listening.

What you will build:

- A Foundry hub plus project in `eastus`, plus a single `gpt-4o-mini` deployment that backs the agent.
- An Azure AI Speech resource (`kind=SpeechServices`, S0 SKU) at `eastus`, with the lab tag and the standard neural voice catalog reachable.
- A Foundry Agent on the Responses API path (no tools), with a system prompt that produces a concise spoken reply under 200 characters, no Markdown.
- A manual Speech STT call on a lab-shipped 12-second WAV, asserting at least 95% character-level similarity to the expected transcript.
- A Voice Live unified loop end to end on the same WAV, capturing both an output WAV and a transcript.

**Time.** Plan for about 60 minutes hands-on. Foundry provisioning is 5 to 8 minutes; Speech provisioning is quick; the STT, agent run, and Voice Live loop each finish in seconds. Session window is 100 minutes including debugging.

**Cost.** A clean run lands at $0.00 to $0.10 per session. Azure Speech STT covers the first 5 audio hours per month on the free tier; this lab pushes 12 seconds through. TTS bills $1 per 1M characters for standard neural voices; a 200-char reply is two-hundredths of a cent. GPT-4o-mini on the agent run is fractions of a cent. Voice Live orchestration itself is free; you pay for the underlying Speech and Azure OpenAI surfaces. A debugging session with a handful of retries still lands well under a dime.

**Voice Live is PREVIEW.** The Voice Live API is in public preview as of beta. The SDK is pinned to `azure-ai-voice-live==0.1.0` so a downstream version bump cannot silently break this lab. If Microsoft ships breaking changes at GA, this lab is the candidate to defer or rebuild. Treat the unified-pipeline pattern as the durable lesson; the exact SDK shape may shift.

This lab maps to AI-103 Domain 4: Implement text analysis solutions (13% of exam weight, including speech).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus Azure SDKs, Speech SDK, and the preview
# Voice Live SDK. The Voice Live SDK is pinned EXACTLY because the surface
# may shift between preview and GA; an unpinned install would silently
# break the Task 4 cell when Microsoft cuts a new beta.

!pip install --quiet labrun-checks==0.6.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 azure-cognitiveservices-speech>=1.40.0 azure-ai-voice-live==0.1.0 openai>=2.0.0 requests>=2.32.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.

import atexit
import difflib
import getpass
import hashlib
import importlib.metadata
import os
import time
import wave

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
import azure.cognitiveservices.speech as speechsdk
import requests

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "voice-live-speech-agent"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
SPEECH_NAME = f"labrun-{LAB_ID}-speech"
AGENT_NAME = f"labrun-{LAB_ID}-agent"

MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30

# Resolved during setup or Task 1.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
PROJECT_ENDPOINT = None
SPEECH_ENDPOINT = None
SPEECH_KEY = None
AZURE_CREDENTIAL = None

# State populated by tasks for downstream checkpoints.
AGENT_ID = None
AGENT_INSTANCE = None
STT_TRANSCRIPT = None
AGENT_REPLY = None
VOICE_LIVE_OUTPUT_WAV_PATH = None
VOICE_LIVE_TRANSCRIPT = None

# Audio constants. The 12-second WAV is hosted in the labrun-labs assets repo
# at a pinned commit so future repo changes cannot break the lab.
SAMPLE_WAV_URL = "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-09/sample-12s.wav"
LOCAL_WAV_PATH = "/tmp/labrun-voice-live-input.wav"
OUTPUT_WAV_PATH = "/tmp/labrun-voice-live-output.wav"
EXPECTED_TRANSCRIPT_SUBSTRING = "what is your return policy"

# Pricing for wallet checks.
SPEECH_STT_FREE_HOURS_PER_MONTH = 5.0
TTS_USD_PER_MILLION_CHARS = 1.0
CHAT_PRICE_PER_1M_INPUT_USD = 0.15
CHAT_PRICE_PER_1M_OUTPUT_USD = 0.60

# Voice Live SDK version pin assertion. The preview SDK shape may shift
# between beta and GA; pinning here makes a silent version drift LOUD.
try:
    _voice_live_version = importlib.metadata.version("azure-ai-voice-live")
except importlib.metadata.PackageNotFoundError:
    print("azure-ai-voice-live is not installed. Re-run the pip cell above.")
    raise SystemExit(1)
if _voice_live_version != "0.1.0":
    print(
        f"Voice Live SDK version {_voice_live_version!r} is not the pinned 0.1.0. "
        f"The preview surface may have shifted; rebuild required before this lab runs."
    )
    raise SystemExit(1)

print("=" * 60)
print("PREVIEW CALLOUT")
print("=" * 60)
print("Voice Live is in public preview as of beta.")
print(f"SDK pinned: azure-ai-voice-live=={_voice_live_version}")
print("This lab may require a rebuild between beta and GA.")
print("Treat the unified-pipeline pattern as durable; the SDK shape may shift.")
print("=" * 60)

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. This lab pins {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# No critical resources: Speech bills only when audio is sent through it,
# the Foundry agent and GPT-4o-mini Standard deployment do not bill at
# zero traffic, and the Voice Live orchestration layer is free. Cleanup
# still tears every resource down for orphan-scan hygiene.
# AGENT_ID is unresolved at declaration; the Task 3 cell rewrites the
# foundry_agent entry once the agent exists, and the cleanup cell resolves
# the <attached-aoai-account> placeholder before run_cleanup runs.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="foundry_agent",
        id="<agent-id-resolved-at-create>",
        region=REGION,
        cli_delete_command=(
            f"az ml agent delete --resource-group {RESOURCE_GROUP} "
            f"--workspace-name {PROJECT_NAME} --agent-id <agent-id-resolved-at-create>"
        ),
    ),
    CleanupResource(
        type="speech_resource",
        id=SPEECH_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account delete "
            f"--resource-group {RESOURCE_GROUP} --name {SPEECH_NAME}"
        ),
    ),
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Provision Foundry, deploy GPT-4o-mini, and stand up an Azure AI Speech resource

The Foundry stack reuses the Lab 01 shape: resource group, hub, project, attached Azure OpenAI account, GPT-4o-mini Standard deployment. The new piece is the Azure AI Speech resource itself (`kind=SpeechServices`), which is the data-plane home for both STT and TTS in this lab.

Build it in this order:

1. Create the resource group, hub, and project with the lab tag.
2. Discover the attached Azure OpenAI account on the hub. Deploy `gpt-4o-mini` at Standard SKU 30k TPM.
3. Create the Azure AI Speech resource: `kind=SpeechServices`, sku `S0`, location `eastus`, tags include `labrun:lab-slug`. Wait for `provisioningState=Succeeded`. Capture the endpoint and pull key1 via `cs_client.accounts.list_keys` for the data-plane TTS and Voice Live calls below.

**Why S0 and not F0.** F0 is rate-limited (20 transactions per minute) and does not support some neural voices. S0 is the standard production tier and the one the AI-103 study guide assumes. The lab cost is still effectively zero because the first 5 audio hours of STT per month are free.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision RG, hub, project, GPT-4o-mini deployment, and the
# Azure AI Speech resource.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()
PROJECT_ENDPOINT = project_workspace.properties.discovery_url
print(f"Project endpoint: {PROJECT_ENDPOINT}")

# Discover the attached Azure OpenAI account.
for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
if not AOAI_ACCOUNT_NAME:
    print("Could not find an attached Azure OpenAI account on the hub.")
    raise SystemExit(1)

aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

# Deploy GPT-4o-mini. This backs the agent in Task 3 and the Voice Live
# loop in Task 4.
chat_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": MODEL_NAME, "version": MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the chat deployment go through Succeeded purgatory, about 1 to 2 minutes...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, chat_payload,
).result()
print(f"GPT-4o-mini deployment ready: {DEPLOYMENT_NAME}")

# Speech resource. kind=SpeechServices, sku=S0, region=eastus, with the
# lab tag so the orphan scan can see it. This is the new piece this lab
# adds on top of the Foundry stack.
speech_payload = {
    "location": REGION,
    "kind": "SpeechServices",
    "sku": {"name": "S0"},
    "tags": lab_tags,
    "properties": {"public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Speech resource, this usually finishes in under a minute...")
# YOUR CODE: Create the Speech resource via
# cs_client.accounts.begin_create_or_update(
#     RESOURCE_GROUP, SPEECH_NAME, speech_payload,
# ).result()
# Store the result in speech_account.

SPEECH_ENDPOINT = speech_account.properties.endpoint
speech_keys = cs_client.accounts.list_keys(RESOURCE_GROUP, SPEECH_NAME)
SPEECH_KEY = speech_keys.key1
print(f"Speech resource provisioned: {SPEECH_NAME}")
print(f"Endpoint: {SPEECH_ENDPOINT}")
print(f"provisioningState: {speech_account.properties.provisioning_state}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Azure AI Speech resource exists at S0 SpeechServices in
# eastus with the lab tag, and the data-plane voices-list endpoint returns
# en-US-AvaNeural in the catalog.

def checkpoint_1(session):
    try:
        try:
            account = cs_client.accounts.get(RESOURCE_GROUP, SPEECH_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Speech account {SPEECH_NAME} not found in {RESOURCE_GROUP}. "
                    f"Did the begin_create_or_update call run and return?"
                ),
            )

        if account.kind != "SpeechServices":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Account kind is {account.kind!r}, expected 'SpeechServices'. "
                    f"This lab pins SpeechServices; CognitiveServices multi-service is the wrong shape."
                ),
            )

        provisioning = getattr(account.properties, "provisioning_state", None)
        if provisioning != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"provisioningState is {provisioning!r}, expected 'Succeeded'. "
                    f"Wait for the poller to complete before re-checking."
                ),
            )

        if account.location != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Speech resource is in {account.location!r}, expected {REGION!r}. "
                    f"Voice Live preview region pinning requires {REGION}."
                ),
            )

        tag_value = (account.tags or {}).get(LAB_TAG_KEY)
        if tag_value != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Speech resource tag {LAB_TAG_KEY}={tag_value!r}, expected {LAB_TAG_VALUE!r}. "
                    f"Orphan scan cannot find the resource without the lab tag."
                ),
            )

        # Data-plane voices-list. Confirms the resource is actually reachable
        # and the standard neural voice catalog is intact.
        if not SPEECH_KEY:
            return CheckpointResult(
                status="fail",
                error_reason="SPEECH_KEY is unset. Pull key1 via cs_client.accounts.list_keys.",
            )
        voices_url = f"https://{REGION}.tts.speech.microsoft.com/cognitiveservices/voices/list"
        try:
            resp = requests.get(
                voices_url,
                headers={"Ocp-Apim-Subscription-Key": SPEECH_KEY},
                timeout=30,
            )
        except requests.RequestException as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not reach Speech voices-list endpoint: {exc}",
            )
        if resp.status_code != 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Speech voices-list returned HTTP {resp.status_code}. "
                    f"Body: {resp.text[:200]!r}"
                ),
            )
        try:
            voices = resp.json()
        except ValueError:
            return CheckpointResult(
                status="fail",
                error_reason=f"voices-list response was not JSON. Body: {resp.text[:200]!r}",
            )
        short_names = {v.get("ShortName") for v in voices if isinstance(v, dict)}
        if "en-US-AvaNeural" not in short_names:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "en-US-AvaNeural is not in the voices catalog returned by the Speech "
                    "resource. The lab requires a standard neural voice; AvaNeural is GA."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which field is wrong. If `kind` is off, you used the wrong account kind (CognitiveServices multi-service does not work for this lab). If `provisioningState` is not `Succeeded`, you did not call `.result()` on the poller. If the tag is missing, the `tags` dict did not land on the payload.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `cs_client.accounts.begin_create_or_update(RESOURCE_GROUP, SPEECH_NAME, speech_payload)`. The `speech_payload` dict already has `kind="SpeechServices"`, `sku.name="S0"`, `location="eastus"`, and the lab tag. The poller is async; call `.result()` and store in `speech_account` so the next lines can read `properties.endpoint` and `provisioning_state`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
speech_account = cs_client.accounts.begin_create_or_update(
    RESOURCE_GROUP, SPEECH_NAME, speech_payload,
).result()
```

</details>

**Wallet check.** Creating the Speech resource is free; you only pay when audio flows through it. The free tier covers the first 5 audio hours of STT per month, which is roughly 18,000 seconds. This lab will push 12. Your morning coffee already cost more than this entire session will.

## Task 2: Submit the lab-shipped WAV to Speech STT and capture the transcript

The lab ships a 12-second WAV that says "What is your return policy if the product was damaged in shipping". The audio is 16-bit PCM at 16 kHz mono, which is the format Speech STT recommends for the highest confidence. You submit it via the Speech SDK and capture the transcript.

Build it in this order:

1. Download the WAV to `/tmp/labrun-voice-live-input.wav`. Verify the format with `wave.open`: framerate 16000, sampwidth 2, nchannels 1. If the format is off, stop and fix it; STT confidence collapses on the wrong format.
2. Construct `speechsdk.SpeechConfig(subscription=SPEECH_KEY, region=REGION)` and a file-based `AudioConfig`. Build a `SpeechRecognizer` and call `recognize_once_async().get()`.
3. Assert the result's `reason` is `ResultReason.RecognizedSpeech` and store the text in `STT_TRANSCRIPT`. Otherwise print the failure detail and re-run.

**Why the substring match.** The seeded audio is engineered to be high-confidence (clear voice, common vocabulary, no background noise). The checkpoint asserts a Levenshtein-ratio similarity of at least 0.95 to the expected substring `what is your return policy`; the trailing clause varies slightly between STT runs but the lead-in is stable.

In [ ]:
# NBVAL_SKIP
# Task 2: Download the WAV, verify the format, submit to Speech STT,
# capture the transcript.

print("Fetching the 12-second WAV from the labrun-labs assets repo...")
wav_resp = requests.get(SAMPLE_WAV_URL, timeout=60)
if wav_resp.status_code != 200:
    print(f"Could not download the sample WAV: HTTP {wav_resp.status_code}")
    raise SystemExit(1)
with open(LOCAL_WAV_PATH, "wb") as f:
    f.write(wav_resp.content)
print(f"WAV saved to {LOCAL_WAV_PATH} ({len(wav_resp.content)} bytes)")

# Verify the format. Wrong format -> STT confidence drops.
with wave.open(LOCAL_WAV_PATH, "rb") as wf:
    framerate = wf.getframerate()
    sampwidth = wf.getsampwidth()
    nchannels = wf.getnchannels()
    nframes = wf.getnframes()
if (framerate, sampwidth, nchannels) != (16000, 2, 1):
    print(
        f"WAV format is framerate={framerate}, sampwidth={sampwidth}, nchannels={nchannels}. "
        f"Speech STT expects 16-bit PCM at 16 kHz mono for the highest confidence."
    )
    raise SystemExit(1)
duration_seconds = nframes / float(framerate)
print(f"WAV format OK: 16-bit PCM, 16 kHz mono, {duration_seconds:.1f} seconds")

# Construct the Speech SDK pipeline.
speech_config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, region=REGION)
audio_config = speechsdk.audio.AudioConfig(filename=LOCAL_WAV_PATH)
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

print("Sending the 12-second WAV to Speech STT, takes about 5 seconds...")
# YOUR CODE: Run the recognizer once and block on the result. Construct
# result = recognizer.recognize_once_async().get()

if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    STT_TRANSCRIPT = result.text
    print(f"Transcript: {STT_TRANSCRIPT!r}")
elif result.reason == speechsdk.ResultReason.NoMatch:
    print("No speech could be recognized in the audio.")
elif result.reason == speechsdk.ResultReason.Canceled:
    cancellation = result.cancellation_details
    print(f"STT canceled: {cancellation.reason}. Error: {cancellation.error_details}")
else:
    print(f"Unexpected STT result.reason: {result.reason}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: STT_TRANSCRIPT is non-empty and matches the expected
# substring with Levenshtein-ratio similarity >= 0.95.

def _ratio(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()


def checkpoint_2(session):
    try:
        if not STT_TRANSCRIPT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "STT_TRANSCRIPT is empty. The Speech STT call did not return "
                    "RecognizedSpeech; check the result.reason in Task 2."
                ),
            )
        transcript_lc = STT_TRANSCRIPT.lower()
        expected_lc = EXPECTED_TRANSCRIPT_SUBSTRING.lower()
        # Compare against the leading substring of the transcript so a longer
        # full sentence still ratios cleanly against the expected lead-in.
        head = transcript_lc[: len(expected_lc) + 8]
        similarity = max(_ratio(transcript_lc, expected_lc), _ratio(head, expected_lc))
        if similarity < 0.95:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Similarity {similarity:.3f} is below the 0.95 threshold. "
                    f"Got transcript {STT_TRANSCRIPT!r}; expected substring "
                    f"{EXPECTED_TRANSCRIPT_SUBSTRING!r}. If the WAV format is not "
                    f"16-bit PCM 16 kHz mono, STT confidence drops."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The format check is brutal: Speech STT expects 16-bit PCM at 16 kHz mono and confidence drops sharply on any other format. If the format check fails, the WAV download is corrupted; re-fetch. If `result.reason` is not `RecognizedSpeech`, look at the cancellation details for the underlying cause (auth, key, region mismatch).

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `result = recognizer.recognize_once_async().get()`. The `recognizer` is already constructed with the audio file and the Speech config; `recognize_once_async` returns a future and `.get()` blocks until the recognizer finishes the file. The result object then exposes `.reason` and `.text`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
result = recognizer.recognize_once_async().get()
```

</details>

**Wallet check.** 12 seconds of STT is well inside the 5-audio-hours-per-month free tier; this call cost $0.00. Even at the paid rate of $1 per audio hour, a 12-second clip is one-third of a cent. Coffee remains undefeated.

## Task 3: Build the Foundry Agent on the Responses API path and run the transcript through it

Voice Live consumes a Foundry Agent on the Responses API path. The deprecated Assistants API surface does NOT integrate with Voice Live; the Checkpoint 3 validator rejects an agent whose class module path traces back to `azure.ai.assistants` or `openai.types.beta.assistant`.

Build it in this order:

1. Construct an `AIProjectClient` against the project endpoint with `DefaultAzureCredential` (no API keys).
2. Create the agent via `client.agents.create_agent(model=DEPLOYMENT_NAME, name=AGENT_NAME, instructions=...)`. The instructions force a concise spoken response under 200 characters and ban Markdown (no asterisks, no backticks, no headers, no underscores). TTS reads those characters literally if they leak through.
3. Create a thread, post `STT_TRANSCRIPT` as a user message, create a run, poll until it terminates, read the latest assistant message into `AGENT_REPLY`.

**Why ban Markdown in the system prompt.** TTS does not strip Markdown. A reply with `**bold**` will be read as "asterisk asterisk bold asterisk asterisk". The prompt-side fix is cheaper than a downstream sanitiser and is the AI-103-documented approach for speech-suitable output.

In [ ]:
# NBVAL_SKIP
# Task 3: Create the Foundry Agent on the Responses API path and run the
# STT transcript through it.

from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=AZURE_CREDENTIAL)

agent_instructions = (
    "You are a customer-support voice agent. Answer the user's question in a "
    "single concise spoken sentence under 200 characters. Output plain speakable "
    "text only. Do not use Markdown of any kind: no asterisks, no backticks, no "
    "headers, no underscores, no bullet points. Do not include URLs or code. If "
    "the question is open-ended, give a direct one-sentence answer that a phone "
    "caller could understand the first time."
)

# YOUR CODE: Create the agent. Construct
# agent = project_client.agents.create_agent(
#     model=DEPLOYMENT_NAME,
#     name=AGENT_NAME,
#     instructions=agent_instructions,
#     metadata={LAB_TAG_KEY: LAB_TAG_VALUE},
# )

AGENT_ID = agent.id
AGENT_INSTANCE = agent
print(f"Agent created: {AGENT_NAME} (id: {AGENT_ID})")

# Resolve the cleanup manifest now that AGENT_ID is known.
for r in CLEANUP_MANIFEST:
    if r.type == "foundry_agent" and "<agent-id-resolved-at-create>" in (r.cli_delete_command or ""):
        r.id = AGENT_ID
        r.cli_delete_command = r.cli_delete_command.replace(
            "<agent-id-resolved-at-create>", AGENT_ID,
        )

# Drive a single turn through the agent on the STT transcript.
print("Asking the agent to give a concise spoken answer...")
thread = project_client.agents.create_thread()
project_client.agents.create_message(
    thread_id=thread.id, role="user", content=STT_TRANSCRIPT,
)
# YOUR CODE: Create the run. Construct
# run = project_client.agents.create_run(thread_id=thread.id, agent_id=AGENT_ID)

delay = 1.0
elapsed = 0.0
while elapsed < 30.0:
    run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)
    if run.status in ("completed", "failed", "cancelled", "expired"):
        break
    time.sleep(delay)
    elapsed += delay
    delay = min(delay * 1.5, 4.0)

if run.status != "completed":
    print(f"Run terminated in status {run.status!r}. Check the project for the error detail.")
    raise SystemExit(1)

AGENT_REPLY = ""
for msg in project_client.agents.list_messages(thread_id=thread.id):
    if msg.role == "assistant":
        if msg.content:
            AGENT_REPLY = msg.content[0].text.value
        break

print(f"Reply: {AGENT_REPLY!r}")
print(f"Reply length: {len(AGENT_REPLY)} chars")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: AGENT_REPLY is non-empty, length <= 200, contains no
# Markdown emphasis/code/heading characters, and the agent class module
# does NOT trace back to the deprecated Assistants surface.

def checkpoint_3(session):
    try:
        if AGENT_INSTANCE is None:
            return CheckpointResult(
                status="fail",
                error_reason="AGENT_INSTANCE is None. Did Task 3 create the agent?",
            )
        module = type(AGENT_INSTANCE).__module__ or ""
        if "assistants" in module:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Agent class lives in {module!r}, which traces back to the deprecated "
                    f"Assistants surface. Voice Live integrates with the Responses API path "
                    f"via AIProjectClient.agents.create_agent only."
                ),
            )

        if not AGENT_REPLY:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "AGENT_REPLY is empty. The run may have ended in a non-completed status, "
                    "or the assistant message could not be parsed from thread.list_messages."
                ),
            )
        if len(AGENT_REPLY) > 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AGENT_REPLY is {len(AGENT_REPLY)} characters; the lab caps spoken "
                    f"replies at 200 so TTS finishes in a reasonable window. Tighten the "
                    f"system prompt to enforce a single concise sentence."
                ),
            )
        banned = ["*", "`", "#", "_"]
        leaked = [c for c in banned if c in AGENT_REPLY]
        if leaked:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AGENT_REPLY contains Markdown-style character(s) {leaked}. TTS reads "
                    f"these literally; tighten the instructions to ban Markdown."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

If the reply leaks Markdown, the system prompt is not strict enough; add an explicit ban on asterisks, backticks, headers, and underscores. If the reply runs over 200 characters, the prompt is asking for too much; enforce a single concise sentence. If the class module check fires, you used `client.beta.assistants` somewhere; switch to `project_client.agents.create_agent`.

</details>

<details><summary>Hint 2 (direction)</summary>

Two calls. `agent = project_client.agents.create_agent(model=DEPLOYMENT_NAME, name=AGENT_NAME, instructions=agent_instructions, metadata={LAB_TAG_KEY: LAB_TAG_VALUE})`. Then `run = project_client.agents.create_run(thread_id=thread.id, agent_id=AGENT_ID)`. The `agent_instructions` string already enforces the speech-suitable output rules; do not weaken them.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
agent = project_client.agents.create_agent(
    model=DEPLOYMENT_NAME,
    name=AGENT_NAME,
    instructions=agent_instructions,
    metadata={LAB_TAG_KEY: LAB_TAG_VALUE},
)
# ...
run = project_client.agents.create_run(thread_id=thread.id, agent_id=AGENT_ID)
```

</details>

**Wallet check.** The agent run spent roughly 300 input tokens (system prompt plus transcript) and 100 output tokens on GPT-4o-mini. At $0.15 / $0.60 per 1M, that is about $0.0001, less than a hundredth of a cent. Coffee mocks us.

## Task 4: Run the Voice Live unified loop end to end on the same WAV

The Voice Live API stitches STT, the agent run, and TTS into a single streaming pipeline. The server-side benefits are real: STT can chunk audio as it arrives, the agent can start responding before STT finishes, and TTS streams audio as it generates. Client-side, you make one call instead of three.

Build it in this order:

1. Import the Voice Live SDK. The import is inside a `try / except ImportError` block because the preview package shape may shift; if the import fails, the message names the pinned version so you do not silently fall off the supported surface.
2. Construct the Voice Live client against the Speech endpoint, the project endpoint, and the agent id. Authenticate with the Speech key.
3. Run the unified pipeline on `LOCAL_WAV_PATH`. Request `output_format="riff-16khz-16bit-mono-pcm"` so the output is a WAV (TTS default is MP3 which breaks the Checkpoint 4 WAV-format assertion).
4. Write `response.audio_bytes` to `OUTPUT_WAV_PATH`. Capture `response.transcript` into `VOICE_LIVE_TRANSCRIPT`.
5. Assert the whole loop finished in under 30 seconds.

**Why a separate STT call earlier if Voice Live already does STT.** The Task 2 STT call is a control measurement so you can see how Voice Live's transcript compares to the standalone STT. They will not be identical character-for-character; the Checkpoint 4 similarity threshold is 0.90, not 0.95, to allow for legitimate differences in chunking and punctuation.

In [ ]:
# NBVAL_SKIP
# Task 4: Voice Live unified loop. STT plus agent plus TTS in one call.

try:
    from azure.ai.voice_live import VoiceLiveClient
except ImportError as exc:
    print(
        "Could not import VoiceLiveClient from azure.ai.voice_live. "
        "This lab pins azure-ai-voice-live==0.1.0 because the preview SDK shape "
        "may shift between beta and GA. Re-run the pip install cell."
    )
    print(f"  Inner error: {exc}")
    raise SystemExit(1)

voice_live_client = VoiceLiveClient(
    speech_endpoint=SPEECH_ENDPOINT,
    speech_key=SPEECH_KEY,
    project_endpoint=PROJECT_ENDPOINT,
    credential=AZURE_CREDENTIAL,
)

print("Synthesizing the response with the AvaNeural voice, exporting to WAV...")
start = time.time()
# YOUR CODE: Run the unified Voice Live pipeline. Construct
# response = voice_live_client.run(
#     audio_file=LOCAL_WAV_PATH,
#     agent_id=AGENT_ID,
#     voice="en-US-AvaNeural",
#     output_format="riff-16khz-16bit-mono-pcm",
# )
elapsed = time.time() - start

with open(OUTPUT_WAV_PATH, "wb") as f:
    f.write(response.audio_bytes)
VOICE_LIVE_OUTPUT_WAV_PATH = OUTPUT_WAV_PATH
VOICE_LIVE_TRANSCRIPT = response.transcript or ""

print(f"Voice Live loop finished in {elapsed:.1f} seconds.")
print(f"Output WAV: {VOICE_LIVE_OUTPUT_WAV_PATH} ({os.path.getsize(VOICE_LIVE_OUTPUT_WAV_PATH)} bytes)")
print(f"Captured transcript: {VOICE_LIVE_TRANSCRIPT!r}")

if elapsed >= 30.0:
    print(f"Loop took {elapsed:.1f}s; Checkpoint 4 caps at 30s. Re-run.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Output WAV exists with non-zero size, 16-bit PCM 16 kHz
# mono, duration between 1 and 15 seconds. Voice Live transcript matches
# the standalone STT transcript at >= 0.90 Levenshtein ratio.

def checkpoint_4(session):
    try:
        if not VOICE_LIVE_OUTPUT_WAV_PATH:
            return CheckpointResult(
                status="fail",
                error_reason="VOICE_LIVE_OUTPUT_WAV_PATH is unset. Did Task 4 run?",
            )
        if not os.path.exists(VOICE_LIVE_OUTPUT_WAV_PATH):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Output WAV file does not exist at {VOICE_LIVE_OUTPUT_WAV_PATH}. "
                    f"Was response.audio_bytes written to disk?"
                ),
            )
        size = os.path.getsize(VOICE_LIVE_OUTPUT_WAV_PATH)
        if size == 0:
            return CheckpointResult(
                status="fail",
                error_reason="Output WAV file is zero bytes. The TTS leg returned empty audio.",
            )

        try:
            with wave.open(VOICE_LIVE_OUTPUT_WAV_PATH, "rb") as wf:
                framerate = wf.getframerate()
                sampwidth = wf.getsampwidth()
                nchannels = wf.getnchannels()
                nframes = wf.getnframes()
        except wave.Error as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Output file is not a valid WAV: {exc}. The TTS default is MP3; "
                    f"pass output_format='riff-16khz-16bit-mono-pcm' to force WAV."
                ),
            )
        if (framerate, sampwidth, nchannels) != (16000, 2, 1):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Output WAV is framerate={framerate}, sampwidth={sampwidth}, "
                    f"nchannels={nchannels}; expected 16000/2/1. Request output_format "
                    f"'riff-16khz-16bit-mono-pcm' on the Voice Live call."
                ),
            )
        duration = nframes / float(framerate)
        if duration < 1.0 or duration > 15.0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Output WAV duration is {duration:.1f}s; expected between 1 and 15s for "
                    f"a sub-200-char spoken reply. Out-of-range duration suggests the agent "
                    f"reply was too long or the TTS leg cut early."
                ),
            )

        if not VOICE_LIVE_TRANSCRIPT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "VOICE_LIVE_TRANSCRIPT is empty. The unified loop did not capture an "
                    "STT transcript from the input audio."
                ),
            )
        sim = difflib.SequenceMatcher(
            None,
            VOICE_LIVE_TRANSCRIPT.lower(),
            (STT_TRANSCRIPT or "").lower(),
        ).ratio()
        if sim < 0.90:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Voice Live transcript similarity {sim:.3f} is below 0.90 vs the standalone "
                    f"STT transcript. Voice Live: {VOICE_LIVE_TRANSCRIPT!r}; STT: "
                    f"{STT_TRANSCRIPT!r}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the output file is not a valid WAV, the TTS leg returned MP3 (the default). If duration is out of range, the agent reply was too long or the TTS cut early; tighten the system prompt. The lab pins `azure-ai-voice-live==0.1.0`; if the import path looks wrong, the preview SDK shape may have shifted and the lab needs a rebuild.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `response = voice_live_client.run(audio_file=LOCAL_WAV_PATH, agent_id=AGENT_ID, voice="en-US-AvaNeural", output_format="riff-16khz-16bit-mono-pcm")`. The `output_format` argument is what forces a WAV; the default is MP3. The client is already constructed with the Speech endpoint, project endpoint, and credential.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
response = voice_live_client.run(
    audio_file=LOCAL_WAV_PATH,
    agent_id=AGENT_ID,
    voice="en-US-AvaNeural",
    output_format="riff-16khz-16bit-mono-pcm",
)
```

</details>

**Wallet check.** Total session damage: a fraction of a cent on GPT-4o-mini for the agent run, free-tier STT for both the standalone and the Voice Live legs, and two-hundredths of a cent on TTS for a sub-200-char reply. Voice Live orchestration itself is free. Your coffee continues to crush the lab cost.

## Cleanup

Time to tear it all down. The cell below deletes the Foundry agent first (cheap, idempotent), then runs the cleanup manifest in reverse-creation order: Speech resource, GPT-4o-mini deployment, project, hub, resource group. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

# Best-effort agent delete before the manifest run. The Azure adapter is a
# stub in labrun-checks today, so the manifest entry will surface as a
# warning; deleting the agent here gives the run an authoritative outcome.
if AGENT_ID and AGENT_ID != "<agent-id-resolved-at-create>":
    try:
        project_client.agents.delete_agent(AGENT_ID)
        print(f"Agent {AGENT_ID} deleted.")
    except Exception as exc:
        print(f"Agent delete skipped: {exc!r}")

# Resolve any remaining placeholders in the manifest.
for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: pennies.** Voice Live is in preview, the SDK is pinned at 0.1.0, and the cost is genuinely pennies. Speech STT is free for the first five audio hours per month, TTS is a buck per million characters, and the agent on GPT-4o-mini barely registers. Coffee remains undefeated. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. The Voice Live API unifies STT, agent, and TTS into a single streaming surface. Walk through what you gain in latency and operational simplicity vs. wiring the three services yourself, AND what you give up in flexibility (custom voice tuning, on-the-fly model swapping, partial-transcript barge-in). When would you prefer the unified API and when would you wire it manually?

2. Voice Live is in public preview. Your team is planning a phone-pilot launch six months from now. Walk through how you would write the architecture so that if Microsoft pulls or rebrands Voice Live between now and launch, the rest of your system survives. Where would you build seams in the code, and what would you log so you can replay against a different speech stack if you have to switch?

## Exam-Style Practice

**Question 1 (Domain 4, Voice Live vs manual wiring):**

A team is building a phone-based customer-support voice agent and wants the lowest end-to-end latency from caller speech to agent voice response. The team is choosing between (i) using Azure Voice Live API as a single streaming pipeline and (ii) wiring Azure Speech STT, a Foundry agent, and Azure Speech TTS as three separate calls. Which choice typically gives the lowest latency, and why?

A. The three-separate-calls approach, because each call can be tuned independently and parallel processing is possible.

B. The Voice Live API, because it streams audio bidirectionally, runs STT and TTS pipelines server-side concurrently with agent inference, and avoids the round-trip overhead of three separate client-server calls.

C. The two are equivalent because the underlying services are the same.

D. The three-separate-calls approach, because the unified API does additional latency-adding safety processing.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: each call adds a serial round-trip on the client side. Even with concurrency tricks on the client, the wall-clock latency is bounded by the slowest leg plus three round-trips.
- B is correct: the Voice Live API is a streaming pipeline. STT chunks the audio as it arrives, the agent can start responding before STT completes, and TTS streams the response audio as it generates. This is the Microsoft-documented latency benefit of the unified API.
- C is wrong: the wire-protocol shape genuinely matters for latency on real-time audio.
- D is wrong: Voice Live includes the same safety filtering surface as the underlying services; it does not add a separate slow safety pass.

</details>

**Question 2 (Domain 4, speech-suitable output):**

A speech agent's assistant reply contains Markdown bullet points, asterisk-bolded phrases, and a backtick-fenced code snippet. The agent's text is being sent to Azure Speech TTS. What happens, and what would you change?

A. TTS reads the Markdown syntax characters out loud (the word "asterisk," the word "backtick"); the fix is to strip Markdown from the assistant reply before sending to TTS, or to prompt the agent to produce speech-suitable output without Markdown.

B. TTS silently strips Markdown and produces clean audio; no change needed.

C. TTS rejects the input as malformed audio prompt; the fix is to base64-encode the Markdown.

D. TTS renders the bullets and bolding as SSML automatically; this is desired behavior.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: TTS reads the Markdown syntax characters literally. This is the documented behavior. The fix is to either post-process the reply to strip Markdown or to prompt the agent to produce speech-suitable text (no Markdown, no code fences, no bullet syntax). Checkpoint 3 in this lab enforces the prompt-side fix.
- B is wrong: TTS does NOT strip Markdown.
- C is wrong: TTS does not reject Markdown text; it just reads it badly.
- D is wrong: TTS does NOT auto-convert Markdown to SSML. SSML is an explicit XML-shaped input format.

</details>